# Fine-tuning con LoRA

El modelo base (Qwen2.5) sabe de todo pero no conoce nuestro dominio específico.
Fine-tuning ajusta el modelo para que aprenda terminología, formato y reglas concretas.

**LoRA (Low-Rank Adaptation)** resuelve el problema del coste:
en lugar de reentrenar todos los pesos del modelo (inviable),
añade matrices pequeñas entrenables encima de los pesos congelados.

- Modelo base: pesos congelados, no cambian
- Matrices LoRA: pequeñas, solo estas se entrenan

El resultado: un modelo especializado con una fracción del coste computacional.

## Dataset — los datos de entrenamiento

Para fine-tuning necesitas ejemplos de input/output que definan
el comportamiento que quieres que aprenda el modelo.
Cuanto más específico y limpio el dataset, mejor el resultado.

Usamos tickets de soporte técnico — el mismo dominio de AI Workflows.

In [ ]:
from datasets import Dataset

# Dataset de entrenamiento — pares input/output
# En producción esto vendría de tus datos reales
data = [
    {
        "input": "Users cannot log in, authentication service returning 500 errors",
        "output": "severity: P1 | area: backend | escalate: true"
    },
    {
        "input": "Button on settings page has wrong color in dark mode",
        "output": "severity: P4 | area: frontend | escalate: false"
    },
    {
        "input": "Database connection pool exhausted, all services affected",
        "output": "severity: P1 | area: database | escalate: true"
    },
    {
        "input": "API gateway returning 429 to enterprise client unexpectedly",
        "output": "severity: P2 | area: backend | escalate: true"
    },
    {
        "input": "Typo in the welcome email subject line",
        "output": "severity: P4 | area: frontend | escalate: false"
    },
    {
        "input": "Payment service down, users cannot complete purchases",
        "output": "severity: P1 | area: backend | escalate: true"
    },
    {
        "input": "Dashboard loads slowly for some users, around 8 seconds",
        "output": "severity: P2 | area: backend | escalate: false"
    },
    {
        "input": "Search results pagination broken on mobile",
        "output": "severity: P3 | area: frontend | escalate: false"
    },
]

dataset = Dataset.from_list(data)
print(f"Dataset: {len(dataset)} ejemplos")
print(f"\nEjemplo:")
print(f"Input:  {dataset[0]['input']}")
print(f"Output: {dataset[0]['output']}")

## Formatear el dataset

El modelo no recibe pares input/output directamente.
Necesita el formato de conversación que usó durante su entrenamiento —
el mismo formato de system/user/assistant que ya conoces.

Cada ejemplo se convierte en una conversación completa
que el modelo aprende a replicar.

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def formatear_ejemplo(ejemplo):
    """
    Convierte un par input/output al formato de conversación del modelo.
    El modelo aprende a replicar exactamente este formato.
    """
    messages = [
        {
            "role": "system",
            "content": "You are a support ticket classifier. Respond only with severity, area and escalation."
        },
        {
            "role": "user",
            "content": ejemplo["input"]
        },
        {
            "role": "assistant",
            "content": ejemplo["output"]
        }
    ]

    texto = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": texto}

dataset_formateado = dataset.map(formatear_ejemplo)

print("Ejemplo formateado:")
print(dataset_formateado[0]["text"])

## Cargar el modelo con LoRA

Cargamos el modelo base y le añadimos las matrices LoRA encima.
Los pesos originales del modelo se congelan — no cambian durante el entrenamiento.
Solo las matrices LoRA pequeñas se actualizan.

### Parámetros de LoRA

**`r` — rango**
Es el tamaño de las matrices LoRA que se añaden.
Piénsalo como la "capacidad de aprendizaje" del fine-tuning.
- `r=4` → matrices muy pequeñas, aprende poco pero consume poca memoria
- `r=8` → equilibrio entre capacidad y memoria (el más común)
- `r=64` → aprende más pero se acerca al coste de fine-tuning completo
Para datasets pequeños como el nuestro, r=8 es más que suficiente.

**`lora_alpha` — escala**
Controla cuánto "peso" tiene el aprendizaje de LoRA respecto al modelo base.
La regla práctica es lora_alpha = r * 2.
Con r=8 → lora_alpha=16. No hay que pensar mucho en esto al principio.

**`target_modules` — qué capas ajustamos**
Un transformer tiene muchas capas internas. No todas necesitan ajustarse.
- `q_proj` → query projection, parte del mecanismo de attention
- `v_proj` → value projection, otra parte del attention
Son las capas que más influyen en cómo el modelo interpreta el contexto.
Ajustar solo estas dos es suficiente para la mayoría de casos de fine-tuning.

**`lora_dropout`**
Regularización — apaga aleatoriamente un 5% de las conexiones durante
el entrenamiento para evitar que el modelo memorice los ejemplos
en lugar de aprender el patrón general (overfitting).
Es el mismo concepto que el dropout en cualquier red neuronal.

**`task_type: CAUSAL_LM`**
Le dice a LoRA que estamos fine-tuneando un modelo de generación de texto.
Existen otros tipos para clasificación, traducción, etc.

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Entrenamiento

El Trainer de HF gestiona el bucle de entrenamiento:
para cada ejemplo del dataset calcula el error entre
lo que generó el modelo y lo que debería haber generado,
y ajusta las matrices LoRA para reducir ese error.

### Parámetros clave

**`num_train_epochs`** — cuántas veces el modelo ve todo el dataset.
Con 8 ejemplos necesitamos más epochs para aprender.

**`per_device_train_batch_size`** — cuántos ejemplos procesa a la vez.
Más alto = más memoria. Con 1 es seguro en cualquier máquina.

**`learning_rate`** — velocidad de ajuste de los pesos.
Demasiado alto = el modelo no converge. Demasiado bajo = aprende muy lento.
2e-4 es el valor estándar para LoRA.

**`save_steps`** — cada cuántos pasos guarda un checkpoint del modelo.

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Tokenizamos el dataset completo
def tokenizar(ejemplo):
    return tokenizer(
        ejemplo["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

dataset_tokenizado = dataset_formateado.map(tokenizar)

# Argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./lora_output",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    save_steps=50,
    logging_steps=10,
    fp16=False,
    report_to="none"
)

# Data collator — gestiona el padding entre ejemplos
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_tokenizado,
    data_collator=data_collator,
)

# Entrenamiento
trainer.train()

## Evaluación — ¿aprendió algo el modelo?

Probamos el modelo fine-tuneado con tickets que NO estaban en el dataset.
Si generalizó correctamente, clasificará tickets nuevos con el formato aprendido.
Si solo memorizó, fallará con inputs ligeramente diferentes.

In [ ]:
model.eval()
model.to("cpu")

def clasificar_ticket(ticket: str) -> str:
    messages = [
        {
            "role": "system",
            "content": "You are a support ticket classifier. Respond only with severity, area and escalation."
        },
        {"role": "user", "content": ticket}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    respuesta = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return respuesta.strip()

# Tickets nuevos — no estaban en el dataset de entrenamiento
tickets = [
    "SSL certificate expired, all HTTPS connections failing",
    "Wrong font size in the footer on mobile",
    "Redis cache completely down, affecting all services",
]

for ticket in tickets:
    resultado = clasificar_ticket(ticket)
    print(f"Ticket: {ticket}")
    print(f"Clasificación: {resultado}")
    print()

## Overfitting — cuando el modelo memoriza en lugar de aprender

El modelo clasificó el ticket de "Wrong font size" como P1 cuando debería ser P4.
Esto es overfitting — uno de los problemas más comunes en machine learning.

### Qué es

El modelo ha memorizado los ejemplos de entrenamiento en lugar de aprender
el patrón general. Es como un estudiante que memoriza las respuestas del examen
de práctica en lugar de entender la materia — si le cambias una palabra, falla.

### Por qué ocurre aquí

Nuestro dataset tiene 8 ejemplos con este desequilibrio:
- P1: 3 ejemplos
- P2: 2 ejemplos
- P3: 1 ejemplo
- P4: 2 ejemplos

Con tan pocos datos y 10 epochs de entrenamiento, el modelo vio cada ejemplo
muchas veces y aprendió que "casi siempre es P1" en lugar de aprender
qué características definen cada severidad.

### Cómo se soluciona

**Más datos** — el remedio principal. Con 200-500 ejemplos balanceados
entre P1/P2/P3/P4 el modelo aprende los patrones reales.

**Dataset balanceado** — igual número de ejemplos por clase.
Si tienes 50 ejemplos P1, necesitas ~50 de P2, P3 y P4.

**Menos epochs** — si el modelo ve los mismos datos demasiadas veces,
los memoriza. Con más datos puedes reducir epochs.

**Early stopping** — parar el entrenamiento cuando el error en datos
de validación empieza a subir aunque el error de entrenamiento baje.
Es la señal de que el modelo está empezando a memorizar.

### La lección

El fine-tuning no es solo código — es datos. El 80% del trabajo
en un proyecto real de fine-tuning es conseguir, limpiar y balancear
el dataset. El modelo es tan bueno como los datos que le das.